<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/Wav2Lip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wav2Lip : synchroniser une vidéo avec un audio

Dans ce notebook, on change encore de logique.

Jusqu'ici, nous avons travaillé surtout sur des images :
- génération ;
- édition ;
- traduction entre domaines.

Maintenant, nous passons à la vidéo et à l'audio.

## Idée principale
On prend :
- une **vidéo** contenant un visage ;
- un **audio** contenant une parole ;

et on produit :
- une **vidéo** où le mouvement des lèvres correspond mieux à cet audio.

## Ce qu'on veut comprendre
- ce qu'est la synchronisation labiale ;
- ce que Wav2Lip modifie dans une vidéo ;
- quelles sont les limites du résultat.

## Où se situe Wav2Lip dans le cours ?

Jusqu'ici :
- **StyleGAN2 / 3** : image générée ;
- **StyleCLIP** : image modifiée avec du texte ;
- **CycleGAN** : image traduite ;
- **Stable Diffusion** : image générée à partir d'un prompt.

Maintenant :
- **Wav2Lip** : une vidéo est modifiée à partir d'un audio.

## Idée clé
Le contrôle principal n'est plus un seed ni un prompt.
Le contrôle principal devient ici :
- l'image du visage dans la vidéo ;
- la parole dans l'audio.

## Ce que fait Wav2Lip

Wav2Lip ne recrée pas toute la vidéo à partir de zéro.

Il essaie surtout de :
- détecter le visage ;
- repérer la zone utile ;
- modifier le mouvement de la bouche ;
- conserver autant que possible le reste de l'image.

## Résultat attendu
Une vidéo où les lèvres semblent prononcer le nouvel audio.

In [ ]:
!nvidia-smi || true

import sys
import platform
import torch

print("Python :", sys.version)
print("Plateforme :", platform.platform())
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Installer et récupérer Wav2Lip

On clone ici le dépôt officiel Wav2Lip
et on installe les dépendances principales.

## Remarque
Le dépôt officiel fournit bien un script `inference.py`
pour synchroniser une vidéo avec un audio.

In [ ]:
!rm -rf /content/Wav2Lip
!git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
%cd /content/Wav2Lip
!pip -q install -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y ffmpeg

## Corriger un problème fréquent avec librosa

Sur des environnements récents, Wav2Lip peut rencontrer
un problème de compatibilité avec `librosa`.

On applique ici une petite correction simple si nécessaire.

In [ ]:
audio_py = "/content/Wav2Lip/audio.py"

with open(audio_py, "r", encoding="utf-8") as f:
    txt = f.read()

old = "return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,"
new = "return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,"

if old in txt:
    txt = txt.replace(old, new)
    with open(audio_py, "w", encoding="utf-8") as f:
        f.write(txt)
    print("Patch librosa appliqué.")
else:
    print("Patch non nécessaire ou déjà appliqué.")

## Télécharger le checkpoint du modèle

Wav2Lip a besoin d'un fichier de poids préentraîné.

## Important
Le dépôt officiel explique bien que l'inférence nécessite un `checkpoint_path`.

Dans ce notebook, on suppose que le fichier est nommé :
`wav2lip_gan.pth`
et qu'il se trouve dans le dossier `checkpoints/`.

In [ ]:
!mkdir -p /content/Wav2Lip/checkpoints

In [ ]:
!wget -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth \
  "https://huggingface.co/rippertnt/wav2lip/resolve/main/checkpoints/wav2lip_gan.pth"

In [ ]:
import os
print("Contenu de checkpoints :")
for f in os.listdir("/content/Wav2Lip/checkpoints"):
    print("-", f)

## Charger la vidéo et l'audio

On téléverse maintenant :
- une vidéo `.mp4` avec un visage visible ;
- un fichier audio `.wav` ou `.mp3`.

## Conseils
Pour une première démonstration :
- vidéo courte ;
- visage assez frontal ;
- bouche visible ;
- audio clair.

## Vidéo Perso

La cellule qui suit n'est à exécuter que si vous avez un mp4 et un mp3 disponibles. Sinon passez à la suivante.

In [ ]:
from google.colab import files

uploaded_media = files.upload()
print("Fichiers reçus :", list(uploaded_media.keys()))
video_candidates = [f for f in uploaded_media.keys() if f.lower().endswith((".mp4"))]
INPUT_VIDEO = uploaded_media[0]
print("Vidéo choisie :", INPUT_VIDEO)

## Télécharger une vidéo de démonstration

Pour éviter les problèmes de fichiers personnels au début, on utilise ici une vidéo publique courte,
avec un visage bien visible et un cadrage simple.

### Pourquoi ce type de vidéo ?
Wav2Lip fonctionne mieux quand :
- le visage est assez frontal ;
- la bouche est visible ;
- il n'y a pas trop de mouvement ;
- la vidéo est courte et lisible.

In [ ]:
VIDEO_URL = "https://www.pexels.com/download/video/9709787/"
VIDEO_PATH = "/content/wav2lip_demo_video.mp4"

!wget -O "$VIDEO_PATH" "$VIDEO_URL"

In [ ]:
from IPython.display import Video, display
display(Video(VIDEO_PATH, embed=True, width=640))

In [ ]:
INPUT_VIDEO = VIDEO_PATH
print("Vidéo choisie :", INPUT_VIDEO)

## Audio Perso
La cellule suivante vous permet de mettre votre propre audio, ne la faites tourner que si vous en avez une sous la main.

In [ ]:
from google.colab import files

uploaded_audio = files.upload()
audio_candidates = [f for f in uploaded_audio.keys() if f.lower().endswith((".wav", ".mp3", ".m4a"))]

if len(audio_candidates) == 0:
    raise ValueError("Aucun fichier audio trouvé.")

INPUT_AUDIO = audio_candidates[0]
print("Audio choisi :", INPUT_AUDIO)

Sinon passez directement ici.

In [ ]:
AUDIO_URL = "https://archive.org/download/JFK_Inaugural_Address_19610120/JFK_Inaugural_Address_19610120.mp3"
AUDIO_PATH = "/content/jfk_inaugural.mp3"

!wget -O "$AUDIO_PATH" "$AUDIO_URL"

In [ ]:
SHORT_AUDIO_PATH = "/content/jfk_short.mp3"

!ffmpeg -y -i /content/jfk_inaugural.mp3 -ss 00:00:45 -t 12 -c copy "$SHORT_AUDIO_PATH"

In [ ]:
INPUT_AUDIO = SHORT_AUDIO_PATH
print("Audio choisi :", INPUT_AUDIO)

## Lancer Wav2Lip

Le dépôt officiel utilise la commande suivante en inférence :

- un `checkpoint_path`
- une vidéo passée avec `--face`
- un audio passé avec `--audio`

Nous allons lancer exactement cette logique.

In [ ]:
CHECKPOINT_PATH = "/content/Wav2Lip/checkpoints/wav2lip_gan.pth"
OUT_PATH = "/content/wav2lip_result.mp4"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint introuvable : {CHECKPOINT_PATH}")

!python /content/Wav2Lip/inference.py \
    --checkpoint_path "$CHECKPOINT_PATH" \
    --face "$INPUT_VIDEO" \
    --audio "$INPUT_AUDIO" \
    --outfile "$OUT_PATH" \
    --face_det_batch_size 4 \
    --wav2lip_batch_size 16
    ##--resize_factor 2

In [ ]:
from IPython.display import Video, display
display(Video("/content/wav2lip_result.mp4", embed=True, width=640))

## Que montre ce notebook ?

Wav2Lip montre qu'un audio peut servir à piloter une partie du visage dans une vidéo.

## Différence avec les notebooks précédents
- ici, on ne génère pas seulement une image ;
- on travaille avec une séquence temporelle ;
- la parole devient un signal de contrôle.

## Idée clé
Le modèle essaie de rendre la bouche compatible avec le son.

## Limites

- la qualité dépend beaucoup de la vidéo d'entrée ;
- une vidéo de profil fonctionne souvent moins bien ;
- des mouvements rapides peuvent dégrader le résultat ;
- certains artefacts apparaissent autour de la bouche ;
- un bon lip-sync ne suffit pas toujours à rendre la vidéo totalement crédible.